# Superposition Analysis

**Superposition** is the phenomenon where neural networks represent more features than they have dimensions,
by encoding features as nearly-orthogonal directions that interfere with each other.

This notebook covers the `irtk.superposition` module:
- Extracting feature directions (PCA, random)
- Measuring feature interference
- Dimensionality analysis via participation ratio
- Activation covariance and sparsity metrics

## Why This Matters

Models represent more features than they have dimensions by encoding them as nearly-orthogonal directions — a phenomenon called superposition. Understanding superposition is essential because it explains why individual neurons are polysemantic and motivates the search for better decomposition methods like sparse autoencoders.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import superposition

# Load GPT-2 Small
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

## 1. Collecting Activations

First, run several sequences through the model and collect activations at a hook point.

In [ ]:
prompts = [
    "The capital of France is Paris, which is known for",
    "Machine learning models can be trained using gradient",
    "The quick brown fox jumped over the lazy dog and",
    "In mathematics, a prime number is a natural number",
    "The stock market experienced significant volatility during the",
    "Scientists recently discovered a new species of deep sea",
    "The history of computing begins with mechanical calculators and",
    "Cooking a perfect steak requires careful attention to temperature",
]

hook_name = "blocks.6.hook_resid_post"  # Mid-layer residual stream
all_activations = []

for prompt in prompts:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens)
    acts = np.array(cache[hook_name])  # [seq_len, d_model]
    all_activations.append(acts)

# Stack all token activations
activations = np.concatenate(all_activations, axis=0)
print(f"Collected activations: {activations.shape}  (tokens x d_model)")

## 2. Feature Directions via PCA

`compute_feature_directions` extracts the principal components of the activation distribution.
These are the directions of maximum variance — the most "important" directions in the space.

In [ ]:
# Extract top 20 PCA directions
pca_dirs = superposition.compute_feature_directions(
    activations, n_features=20, method="pca"
)
print(f"PCA directions shape: {pca_dirs.shape}")
print(f"Norms (should be 1.0): {np.linalg.norm(pca_dirs, axis=-1)[:5]}")

# Verify orthogonality
gram = pca_dirs @ pca_dirs.T
print(f"Max off-diagonal (should be ~0): {np.max(np.abs(gram - np.eye(20))):.6f}")

In [ ]:
# Compare with random directions
rand_dirs = superposition.compute_feature_directions(
    activations, n_features=20, method="random"
)
print(f"Random directions shape: {rand_dirs.shape}")
print(f"Norms: {np.linalg.norm(rand_dirs, axis=-1)[:5]}")

## 3. Feature Interference

When features are nearly-orthogonal but not exactly orthogonal, they **interfere** with each other.
`feature_interference` computes the pairwise cosine similarity matrix between feature directions.

In [ ]:
# PCA directions are orthogonal, so no interference
pca_interference = superposition.feature_interference(pca_dirs)
print(f"PCA interference (off-diagonal max): {np.max(np.abs(pca_interference - np.diag(np.diag(pca_interference)))):.6f}")

# Random directions in high-D space are nearly orthogonal
rand_interference = superposition.feature_interference(rand_dirs)
off_diag = rand_interference - np.diag(np.diag(rand_interference))
print(f"Random interference (off-diagonal mean abs): {np.mean(np.abs(off_diag)):.4f}")
print(f"Random interference (off-diagonal max abs): {np.max(np.abs(off_diag)):.4f}")

In [ ]:
# Visualize interference matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(pca_interference, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("PCA Feature Interference")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Feature")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(rand_interference, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("Random Feature Interference")
axes[1].set_xlabel("Feature")
axes[1].set_ylabel("Feature")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

## 4. Dimensionality Analysis

The **participation ratio** measures the effective dimensionality of activations at each layer.
If activations use all dimensions equally, PR = d_model. If activations cluster along a few axes, PR is low.

PR = (sum of eigenvalues)^2 / sum(eigenvalues^2)

In [ ]:
# Run dimensionality analysis across layers
token_seqs = [model.to_tokens(p) for p in prompts]
dim_result = superposition.dimensionality_analysis(model, token_seqs)

print(f"Layers analyzed: {dim_result['labels']}")
print(f"Participation ratios: {dim_result['participation_ratio']}")

In [ ]:
# Plot participation ratio across layers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = dim_result["labels"]
pr = dim_result["participation_ratio"]
axes[0].bar(range(len(labels)), pr)
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=45, ha="right")
axes[0].set_ylabel("Participation Ratio")
axes[0].set_title("Effective Dimensionality by Layer")
axes[0].axhline(model.cfg.d_model, color="red", linestyle="--", alpha=0.5, label=f"d_model={model.cfg.d_model}")
axes[0].legend()

# Plot eigenvalue spectra for embed + a few layers
for i, (label, spectrum) in enumerate(zip(labels, dim_result["eigenvalue_spectra"])):
    if i % 4 == 0 or i == len(labels) - 1:  # Plot every 4th + last
        axes[1].plot(spectrum[:50] / spectrum[0], label=label, alpha=0.8)

axes[1].set_xlabel("Component Index")
axes[1].set_ylabel("Normalized Eigenvalue")
axes[1].set_title("Eigenvalue Spectra (top 50)")
axes[1].legend(fontsize=8)
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

## 5. Activation Covariance

The covariance matrix reveals how activation dimensions correlate with each other.
In a model using superposition, off-diagonal correlations indicate feature interference.

In [ ]:
# Covariance at an early layer vs a late layer
cov_early = superposition.activation_covariance(model, token_seqs, "blocks.0.hook_resid_post")
cov_late = superposition.activation_covariance(model, token_seqs, "blocks.10.hook_resid_post")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

vmax = max(np.abs(cov_early).max(), np.abs(cov_late).max())
im0 = axes[0].imshow(cov_early[:32, :32], cmap="RdBu_r", vmin=-vmax/5, vmax=vmax/5)
axes[0].set_title("Layer 0 Covariance (32x32 block)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(cov_late[:32, :32], cmap="RdBu_r", vmin=-vmax/5, vmax=vmax/5)
axes[1].set_title("Layer 10 Covariance (32x32 block)")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

## 6. Feature Sparsity

Sparsity is a key ingredient for superposition — features that are rarely active can be packed
into fewer dimensions with less interference.

`feature_sparsity` computes multiple sparsity metrics:
- **L0 fraction**: Fraction of activations above threshold
- **Kurtosis**: High kurtosis = heavy tails = sparse
- **Gini coefficient**: 0 = uniform, 1 = maximally sparse

In [ ]:
# Measure sparsity of collected activations
sparsity = superposition.feature_sparsity(activations)
print(f"L0 mean (avg active dims per sample): {sparsity['l0_mean']:.1f}")
print(f"L0 fraction: {sparsity['l0_fraction']:.3f}")
print(f"Kurtosis mean: {sparsity['kurtosis_mean']:.2f}")
print(f"Gini mean: {sparsity['gini_mean']:.3f}")

In [ ]:
# Compare sparsity across layers
layer_sparsity = []
layer_names = []

for layer_idx in range(model.cfg.n_layers):
    hook = f"blocks.{layer_idx}.hook_resid_post"
    layer_acts = []
    for tokens in token_seqs:
        _, cache = model.run_with_cache(tokens)
        layer_acts.append(np.array(cache[hook]))
    layer_acts = np.concatenate(layer_acts, axis=0)
    sp = superposition.feature_sparsity(layer_acts)
    layer_sparsity.append(sp)
    layer_names.append(f"L{layer_idx}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = range(len(layer_names))

axes[0].plot(x, [s["l0_fraction"] for s in layer_sparsity], "o-")
axes[0].set_title("L0 Fraction by Layer")
axes[0].set_xlabel("Layer")

axes[1].plot(x, [s["kurtosis_mean"] for s in layer_sparsity], "o-", color="orange")
axes[1].set_title("Kurtosis by Layer")
axes[1].set_xlabel("Layer")

axes[2].plot(x, [s["gini_mean"] for s in layer_sparsity], "o-", color="green")
axes[2].set_title("Gini Coefficient by Layer")
axes[2].set_xlabel("Layer")

plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `compute_feature_directions()` | Extract feature directions via PCA or random projections |
| `feature_interference()` | Pairwise cosine similarity between directions |
| `dimensionality_analysis()` | Participation ratio and eigenvalue spectra per layer |
| `activation_covariance()` | Covariance matrix at any hook point |
| `feature_sparsity()` | L0, kurtosis, and Gini sparsity metrics |